# YouTube + Google Meet QoE Report

A per-tier rendering of the same QoE graphs used in `pramana_demo.ipynb` for the synchronized 15-second YouTube + Google Meet experiment with 100 ms latency and pfifo.

In [ ]:
from pathlib import Path
import json
import os
import math
import subprocess

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

BANDWIDTH_MBPS = int(os.environ.get('BANDWIDTH_MBPS', '3'))
result_candidates = [
    Path(f'youtube_google_meet_100ms_pfifo/results/{BANDWIDTH_MBPS}mbps'),
    Path(f'experiments/youtube_google_meet_100ms_pfifo/results/{BANDWIDTH_MBPS}mbps'),
]
RESULT_DIR = next((p.resolve() for p in result_candidates if p.is_dir()), None)
if RESULT_DIR is None:
    raise FileNotFoundError(f'Could not locate the {BANDWIDTH_MBPS} Mbps result directory')

def load_qoe(app):
    path = RESULT_DIR / f'{app}_stats.jsonl'
    if not path.is_file():
        raise FileNotFoundError(f'Missing required QoE data: {path}')
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

youtube_qoe = load_qoe('youtube')
meet_qoe = load_qoe('google_meet')
start_time = min(youtube_qoe[0]['timestamp'], meet_qoe[0]['timestamp'])
youtube_seconds = [row['timestamp'] - start_time for row in youtube_qoe]
meet_seconds = [row['timestamp'] - start_time for row in meet_qoe]
plt.style.use('seaborn-v0_8-whitegrid')

ys = [row['stats'] for row in youtube_qoe]
ms = [row['stats'] for row in meet_qoe]
meet_rates = [s['inbound_bitrate_mbps'] for s in ms if s.get('inbound_bitrate_mbps') is not None]
summary = pd.DataFrame([{
    'Bandwidth': f'{BANDWIDTH_MBPS} Mbps',
    'YouTube final resolution': ys[-1]['resolution'],
    'YouTube dropped frames': (ys[-1].get('dropped_video_frames', 0) or 0) - (ys[0].get('dropped_video_frames', 0) or 0),
    'Meet final resolution': ms[-1]['resolution'],
    'Meet avg video Mbps': sum(meet_rates) / len(meet_rates),
    'Meet dropped frames': ms[-1]['frames_dropped'] - ms[0]['frames_dropped'],
    'Meet new freezes': ms[-1]['freeze_count'] - ms[0]['freeze_count'],
}]).set_index('Bandwidth')
display(summary.round(3))

## Per-application download traffic

Traffic is attributed from TLS/QUIC server names and Meet TURN media ports. As a requested experiment-specific heuristic, the highest-volume otherwise-unclassified `172.217.*` flow is assigned to Google Meet, the second-highest to YouTube, and all remaining unclassified inbound traffic is assigned to Google Meet. These assignments are assumptions, not packet-level ground truth.

In [ ]:
PCAP = next(RESULT_DIR.glob('*.pcap'))
CLIENT_IP = '172.16.1.1'
window_start = min(youtube_qoe[0]['timestamp'], meet_qoe[0]['timestamp'])
window_end = max(youtube_qoe[-1]['timestamp'], meet_qoe[-1]['timestamp'])
window_seconds = window_end - window_start

sni_output = subprocess.run(
    ['tshark', '-r', str(PCAP), '-Y', 'tls.handshake.extensions_server_name',
     '-T', 'fields', '-e', 'ip.dst', '-e', 'tls.handshake.extensions_server_name'],
    check=True, capture_output=True, text=True,
).stdout
hosts_by_ip = {}
for line in sni_output.splitlines():
    fields = line.split('\t')
    if len(fields) >= 2 and fields[0] and fields[1]:
        hosts_by_ip.setdefault(fields[0], set()).update(h.lower() for h in fields[1].split(','))

youtube_markers = ('youtube.com', 'googlevideo.com', 'ytimg.com', 'ggpht.com', 'gvt1.com')
meet_markers = ('meet.google.com', 'hangouts.clients6.google.com')
youtube_ips = {ip for ip, hosts in hosts_by_ip.items() if any(marker in host for host in hosts for marker in youtube_markers)}
meet_ips = {ip for ip, hosts in hosts_by_ip.items() if any(marker in host for host in hosts for marker in meet_markers)} - youtube_ips

packet_output = subprocess.run(
    ['tshark', '-r', str(PCAP), '-Y', f'ip.dst == {CLIENT_IP}', '-T', 'fields',
     '-e', 'frame.time_epoch', '-e', 'ip.src', '-e', 'udp.srcport', '-e', 'tcp.srcport', '-e', 'frame.len'],
    check=True, capture_output=True, text=True,
).stdout
packet_rows = []
for line in packet_output.splitlines():
    fields = line.split('\t')
    if len(fields) < 5 or not fields[0] or not fields[1] or not fields[4]:
        continue
    timestamp = float(fields[0])
    if not window_start <= timestamp <= window_end:
        continue
    remote_ip = fields[1]
    port_text = fields[2] or fields[3] or '0'
    remote_port = int(port_text.split(',')[0])
    frame_bytes = int(fields[4].split(',')[0])
    if remote_ip in youtube_ips:
        app = 'YouTube'
    elif remote_ip in meet_ips or 19302 <= remote_port <= 19309:
        app = 'Google Meet'
    else:
        app = 'Unclassified'
    packet_rows.append((timestamp, remote_ip, frame_bytes, app))

packets = pd.DataFrame(packet_rows, columns=['timestamp', 'remote_ip', 'frame_bytes', 'application'])
assumption_candidates = (packets[
    (packets.application == 'Unclassified') & packets.remote_ip.str.startswith('172.217.')
].groupby('remote_ip').frame_bytes.sum().sort_values(ascending=False).head(2))
assumed_ip_map = {}
if len(assumption_candidates) >= 1:
    assumed_ip_map[assumption_candidates.index[0]] = 'Google Meet'
if len(assumption_candidates) >= 2:
    assumed_ip_map[assumption_candidates.index[1]] = 'YouTube'
if assumed_ip_map:
    assumed_rows = []
    for remote_ip, app in assumed_ip_map.items():
        assumed_rows.append({
            'remote IP': remote_ip,
            'assumed application': app,
            'inbound download (MiB)': assumption_candidates[remote_ip] / 2**20,
        })
        packets.loc[packets.remote_ip == remote_ip, 'application'] = app
    print('Requested heuristic attribution (assumption, not packet-level ground truth):')
    display(pd.DataFrame(assumed_rows).set_index('remote IP').round(3))
remaining_assumed_meet_bytes = packets.loc[packets.application == 'Unclassified', 'frame_bytes'].sum()
packets.loc[packets.application == 'Unclassified', 'application'] = 'Google Meet'
print(f'All remaining previously-unclassified inbound traffic assigned to Google Meet: {remaining_assumed_meet_bytes / 2**20:.3f} MiB')
packets['second'] = (packets.timestamp - window_start).astype(int)
bin_count = max(1, math.ceil(window_seconds))
applications = ['YouTube', 'Google Meet']
app_bytes = (packets[packets.application.isin(applications)]
             .groupby(['second', 'application']).frame_bytes.sum()
             .unstack(fill_value=0)
             .reindex(range(bin_count), fill_value=0)
             .reindex(columns=applications, fill_value=0))
app_mbps = app_bytes * 8 / 1_000_000
attributed_bytes = app_bytes.to_numpy().sum()
summary_rows = []
for app in applications:
    total_bytes = app_bytes[app].sum()
    summary_rows.append({
        'application': app,
        'download (MiB)': total_bytes / 2**20,
        'average during QoE window (Mbps)': total_bytes * 8 / window_seconds / 1_000_000,
        'peak 1-second bin (Mbps)': app_mbps[app].max(),
        'share of attributed bytes (%)': 100 * total_bytes / attributed_bytes if attributed_bytes else 0,
    })
display(pd.DataFrame(summary_rows).set_index('application').round(3))

fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
ax.plot(app_mbps.index, app_mbps['YouTube'], marker='o', color='#ff0000', label='YouTube')
ax.plot(app_mbps.index, app_mbps['Google Meet'], marker='o', color='#1f78b4', label='Google Meet')
ax.axhline(BANDWIDTH_MBPS, color='#333333', linestyle='--', alpha=.7, label=f'Configured bottleneck ({BANDWIDTH_MBPS} Mbps)')
ax.set(title='Per-application download traffic during concurrent playback', xlabel='Seconds from first QoE sample', ylabel='Downloaded Mbit in each 1-second bin')
ax.set_xticks(range(bin_count))
ax.legend()
plt.tight_layout()
plt.show()

print(f'Attribution window: {window_seconds:.3f} seconds')
print(f'App-attributed download after requested assumptions: {attributed_bytes / 2**20:.3f} MiB')

## YouTube playback health

In [ ]:
youtube_buffer = [row['stats'].get('buffer_ahead_secs', 0) or 0 for row in youtube_qoe]
youtube_drop_pct = [
    100 * (row['stats'].get('dropped_video_frames', 0) or 0)
    / max(1, row['stats'].get('total_video_frames', 0) or 0)
    for row in youtube_qoe
]
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
drop_ax = ax.twinx()
buffer_line = ax.plot(youtube_seconds, youtube_buffer, marker='o', color='#ff0000', label='Buffer ahead')[0]
drop_line = drop_ax.plot(youtube_seconds, youtube_drop_pct, marker='s', color='#6a3d9a', label='Dropped frames')[0]
ax.axhline(0, color='#ff7f7f', linestyle='--', alpha=.7)
ax.set(title=f'YouTube Playback Health — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Buffer ahead (seconds)')
drop_ax.set_ylabel('Cumulative dropped frames (%)')
ax.legend([buffer_line, drop_line], ['Buffer ahead', 'Dropped frames'], loc='best')
plt.tight_layout()
plt.show()

## Video resolution over time

In [ ]:
youtube_width = [row['stats'].get('video_width', 0) or 0 for row in youtube_qoe]
youtube_height = [row['stats'].get('video_height', 0) or 0 for row in youtube_qoe]
meet_width = [row['stats'].get('frame_width', 0) or 0 for row in meet_qoe]
meet_height = [row['stats'].get('frame_height', 0) or 0 for row in meet_qoe]

fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.step(youtube_seconds, youtube_height, where='post', color='#ff0000', linewidth=2, label='YouTube')
ax.scatter(youtube_seconds, youtube_height, color='#ff0000', s=24)
ax.step(meet_seconds, meet_height, where='post', color='#1f78b4', linewidth=2, label='Google Meet')
ax.scatter(meet_seconds, meet_height, color='#1f78b4', s=24)

resolution_labels = {}
for width, height in zip(youtube_width + meet_width, youtube_height + meet_height):
    if width and height:
        resolution_labels.setdefault(height, set()).add(f'{width}x{height}')
tick_heights = sorted(resolution_labels)
if tick_heights:
    ax.set_yticks(tick_heights, [' / '.join(sorted(resolution_labels[h])) for h in tick_heights])
ax.set(
    title=f'Video Resolution Over Time — {BANDWIDTH_MBPS} Mbps',
    xlabel='Seconds from synchronized start',
    ylabel='Received/rendered resolution',
)
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## Google Meet receiver network QoE

In [ ]:
meet_bitrate = [row['stats'].get('inbound_bitrate_mbps', 0) or 0 for row in meet_qoe]
meet_jitter_ms = [1000 * (row['stats'].get('jitter_seconds', 0) or 0) for row in meet_qoe]
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
jitter_ax = ax.twinx()
bitrate_line = ax.plot(meet_seconds, meet_bitrate, marker='o', color='#1f78b4', label='Inbound video bitrate')[0]
jitter_line = jitter_ax.plot(meet_seconds, meet_jitter_ms, marker='s', color='#ff7f00', label='RTP jitter')[0]
ax.set(title=f'Google Meet Receiver Network QoE — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Inbound video bitrate (Mbps)')
jitter_ax.set_ylabel('RTP jitter (ms)')
ax.legend([bitrate_line, jitter_line], ['Inbound video bitrate', 'RTP jitter'], loc='best')
plt.tight_layout()
plt.show()

## Google Meet video delivery QoE

In [ ]:
decoded_fps = [row['stats'].get('frames_decoded_delta', 0) or 0 for row in meet_qoe]
height = [row['stats'].get('frame_height', 0) or 0 for row in meet_qoe]
dropped = [row['stats'].get('frames_dropped_delta', 0) or 0 for row in meet_qoe]
freezes = [row['stats'].get('freeze_count_delta', 0) or 0 for row in meet_qoe]
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
resolution_ax = ax.twinx()
fps_line = ax.plot(meet_seconds, decoded_fps, marker='o', color='#33a02c', label='Decoded frames/sample')[0]
drop_line = ax.plot(meet_seconds, dropped, marker='x', color='#e31a1c', label='Dropped frames/sample')[0]
freeze_line = ax.plot(meet_seconds, freezes, marker='D', color='#6a3d9a', label='New freezes/sample')[0]
resolution_line = resolution_ax.step(meet_seconds, height, where='post', color='#1f78b4', alpha=.65, label='Received height')[0]
ax.set(title=f'Google Meet Video Delivery QoE — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Frames per sample interval')
resolution_ax.set_ylabel('Received video height (pixels)')
ax.legend([fps_line, drop_line, freeze_line, resolution_line], ['Decoded frames/sample', 'Dropped frames/sample', 'New freezes/sample', 'Received height'], loc='best')
plt.tight_layout()
plt.show()